In [1]:
# Installation
!pip install ultralytics -q
!pip install editdistance -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 22.1 MB/s eta 0:00:00


In [2]:
# Imports
import os
import shutil
import random
import yaml
import zipfile
from ultralytics import YOLO
import cv2
from tqdm import tqdm
import matplotlib.pyplot as plt
import torch
import gc
import editdistance
import torch.nn as nn
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import glob
import re
import numpy as np

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


**PART A**

In [3]:
path= '/content/NLP_Data.zip'
extractedPath= '/content/ExtractedData'

with zipfile.ZipFile(path, 'r') as zip_ref:
    zip_ref.extractall(extractedPath)

images, labels= None, None
for root, dirs, files in os.walk(extractedPath):
    if 'CheckImages' in dirs:
        images= os.path.join(root, 'CheckImages')
    if 'BoundingBoxes' in dirs:
        labels= os.path.join(root, 'BoundingBoxes')

YOLO_Path= "/content/YOLO_Dataset"
dirs= {
    "train_images": os.path.join(YOLO_Path, "images", "train"),
    "val_images": os.path.join(YOLO_Path, "images", "val"),
    "train_labels": os.path.join(YOLO_Path, "labels", "train"),
    "val_labels": os.path.join(YOLO_Path, "labels", "val")
}

for d in dirs.values():
    os.makedirs(d, exist_ok=True)

all_files = [f for f in os.listdir(images) if f.endswith(('.png', '.jpg', '.tif'))]
random.seed(42)
random.shuffle(all_files)

split_idx = int(len(all_files) * 0.8)
train_files, val_files = all_files[:split_idx], all_files[split_idx:]

def copy_files(files, img_dest, lbl_dest):
    for f in files:
        base_name = os.path.splitext(f)[0]
        shutil.copy(os.path.join(images, f), os.path.join(img_dest, f))
        lbl_file = base_name + ".txt"
        if os.path.exists(os.path.join(labels, lbl_file)):
            shutil.copy(os.path.join(labels, lbl_file), os.path.join(lbl_dest, lbl_file))

copy_files(train_files, dirs["train_images"], dirs["train_labels"])
copy_files(val_files, dirs["val_images"], dirs["val_labels"])

yaml_content = {
    "path": YOLO_Path,
    "train": "images/train",
    "val": "images/val",
    "nc": 2,
    "names": ["legal_amount", "courtesy_amount"]
}
with open(os.path.join(YOLO_Path, "data.yaml"), 'w') as f:
    yaml.dump(yaml_content, f)

print(f"Train: {len(train_files)} images, Val: {len(val_files)} images.")

Train: 1440 images, Val: 360 images.


In [4]:
images_dir_train = "/content/YOLO_Dataset/images/train"
images_dir_val = "/content/YOLO_Dataset/images/val"
for directory in [images_dir_train, images_dir_val]:
    if not os.path.exists(directory):
        continue

    files = os.listdir(directory)
    for filename in tqdm(files, desc=f"Processing {os.path.basename(directory)}"):
        if filename.endswith(('.tif', '.png', '.jpg')):
            filepath = os.path.join(directory, filename)
            img = cv2.imread(filepath)

            if img is not None:
                cv2.imwrite(filepath, img)

Processing val: 100%|██████████| 360/360 [00:08<00:00, 40.65it/s]


In [5]:
model = YOLO('yolov8n.pt')
results = model.train(
    data="/content/YOLO_Dataset/data.yaml",
    epochs=30,
    imgsz=640,
    batch=16,
    name="check_extraction"
)

Ultralytics 8.4.48 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/YOLO_Dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=check_extraction, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, 

In [ ]:
def yolo_to_corners(yolo_box, img_w, img_h):
    x_c, y_c, w, h = yolo_box
    return [int((x_c - w/2) * img_w), int((y_c - h/2) * img_h),
            int((x_c + w/2) * img_w), int((y_c + h/2) * img_h)]
def calculate_iou(box1, box2):
    b1_x1, b1_y1, b1_x2, b1_y2 = box1
    b2_x1, b2_y1, b2_x2, b2_y2 = box2

    inter_x1 = max(b1_x1, b2_x1)
    inter_y1 = max(b1_y1, b2_y1)
    inter_x2 = min(b1_x2, b2_x2)
    inter_y2 = min(b1_y2, b2_y2)
    inter_area = max(0, inter_x2 - inter_x1) * max(0, inter_y2 - inter_y1)
    union_area = ((b1_x2 - b1_x1) * (b1_y2 - b1_y1)) + ((b2_x2 - b2_x1) * (b2_y2 - b2_y1)) - inter_area
    return inter_area / union_area if union_area > 0 else 0
weight_files = glob.glob('/content/runs/detect/**/weights/best.pt', recursive=True)

latest_weights= max(weight_files, key=os.path.getctime)
model = YOLO(latest_weights)
val_images_dir= "/content/YOLO_Dataset/images/val"
val_labels_dir= "/content/YOLO_Dataset/labels/val"

results_data= []

for img_name in os.listdir(val_images_dir):
    img_path = os.path.join(val_images_dir, img_name)
    lbl_path = os.path.join(val_labels_dir, os.path.splitext(img_name)[0] + ".txt")

    img = cv2.imread(img_path)
    h, w, _ = img.shape
    gt_boxes = {}
    if os.path.exists(lbl_path):
        with open(lbl_path, 'r') as f:
            for line in f:
                parts = [float(x) for x in line.strip().split()]
                gt_boxes[int(parts[0])] = yolo_to_corners(parts[1:], w, h)
    preds = model.predict(img_path, verbose=False)[0]
    pred_boxes = {}
    for box in preds.boxes:
        cls_id = int(box.cls[0].item())
        pred_boxes[cls_id] = yolo_to_corners(box.xywhn[0].tolist(), w, h)
    iou_legal = calculate_iou(gt_boxes.get(0, [0,0,0,0]), pred_boxes.get(0, [0,0,0,0])) if 0 in gt_boxes else -1
    iou_courtesy = calculate_iou(gt_boxes.get(1, [0,0,0,0]), pred_boxes.get(1, [0,0,0,0])) if 1 in gt_boxes else -1
    results_data.append({
        'image': img_name, 'img_path': img_path,
        'iou_legal': iou_legal, 'iou_courtesy': iou_courtesy,
        'gt': gt_boxes, 'pred': pred_boxes
    })
valid_legal = [x['iou_legal'] for x in results_data if x['iou_legal'] >= 0]
valid_courtesy = [x['iou_courtesy'] for x in results_data if x['iou_courtesy'] >= 0]
print("legal and courtesy amount regions evaluation:")
print(f"Overall Mean IoU (Legal): {sum(valid_legal)/len(valid_legal)*100:.2f}%")
print(f"Overall Mean IoU (Courtesy): {sum(valid_courtesy)/len(valid_courtesy)*100:.2f}%\n")
for name, data in [("Legal Amount", valid_legal), ("Courtesy Amount", valid_courtesy)]:
    if not data: continue
    acc_50 = sum(1 for i in data if i >= 0.50) / len(data) * 100
    acc_75 = sum(1 for i in data if i >= 0.75) / len(data) * 100
    acc_90 = sum(1 for i in data if i >= 0.90) / len(data) * 100
    print(f"{name} -> Acc@50: {acc_50:.2f}% | Acc@75: {acc_75:.2f}% | Acc@90: {acc_90:.2f}%")
print("WORST CASES")
def plot_worst_cases(class_id, class_name, data):
    worst = sorted([x for x in data if x[f'iou_{class_name.split()[0].lower()}'] >= 0],
                   key=lambda x: x[f'iou_{class_name.split()[0].lower()}'])[:3]

    if not worst: return

    fig, axes = plt.subplots(len(worst), 1, figsize=(10, 5 * len(worst)))
    if len(worst) == 1: axes = [axes]

    fig.suptitle(f"Worst Cases: {class_name}\nGreen = Ground Truth | Red = Prediction", fontsize=14)

    for ax, item in zip(axes, worst):
        img = cv2.imread(item['img_path'])
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        if class_id in item['gt']:
            b = item['gt'][class_id]
            cv2.rectangle(img, (b[0], b[1]), (b[2], b[3]), (0, 255, 0), 3)
        if class_id in item['pred']:
            b = item['pred'][class_id]
            cv2.rectangle(img, (b[0], b[1]), (b[2], b[3]), (255, 0, 0), 3)

        ax.imshow(img)
        ax.set_title(f"File: {item['image']} | IoU: {item[f'iou_{class_name.split()[0].lower()}']:.2f}")
        ax.axis('off')
    plt.tight_layout()
    plt.show()

plot_worst_cases(0, "Legal Amount", results_data)
plot_worst_cases(1, "Courtesy Amount", results_data)


In [ ]:
#the above cell output was removed for privacy reasons

In [7]:
if 'model' in locals():
    del model
gc.collect()
torch.cuda.empty_cache()

**PART B**

In [30]:
extract_path= '/content/ExtractedData'
images_dir= os.path.join(extract_path, 'CheckImages')
labels_dir= os.path.join(extract_path, 'BoundingBoxes')
courtesy_txt_dir= os.path.join(extract_path, 'CourtesyAmounts')
corps= "/content/Courtesy_Crops"
os.makedirs(corps, exist_ok=True)
amount_dict= {}
txt_files= glob.glob(os.path.join(courtesy_txt_dir, '*.txt'))
for txt_file in txt_files:
    with open(txt_file, 'r', encoding='utf-8') as f:
        for line in f:
            if not line.strip(): continue
            parts = line.strip().split(maxsplit=1)
            if len(parts) >= 2:
                filename = parts[0].strip()
                token_str = parts[1].strip()
                tokens = token_str.strip('[]').replace("'", "").split(',')
                digits = [t.strip() for t in tokens if t.strip() not in ['10', '.', '']]
                clean_amount = "".join(digits)
                amount_dict[filename] = clean_amount

processed_data= []
missing_files= 0

for raw_name, text_label in tqdm(amount_dict.items()):
    actual_name = raw_name
    if actual_name.startswith('Cac'):
        actual_name = 'ac' + actual_name[3:]
    img_path = os.path.join(images_dir, actual_name)
    bbox_name = os.path.splitext(actual_name)[0] + '.txt'
    bbox_path = os.path.join(labels_dir, bbox_name)
    if not os.path.exists(img_path):
        base = os.path.splitext(actual_name)[0]
        for ext in ['.png', '.jpg', '.jpeg']:
            alt = os.path.join(images_dir, base + ext)
            if os.path.exists(alt):
                img_path = alt
                actual_name = base + ext
                break

    if not os.path.exists(img_path) or not os.path.exists(bbox_path):
        missing_files += 1
        continue

    img= cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    if img is None: continue
    h, w= img.shape
    with open(bbox_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if not parts: continue
            try:
                class_id = int(float(parts[0]))
                if class_id == 1:
                    x_c, y_c, bw, bh = [float(x) for x in parts[1:]]
                    x1 = max(0, int((x_c - bw/2) * w))
                    y1 = max(0, int((y_c - bh/2) * h))
                    x2 = min(w, int((x_c + bw/2) * w))
                    y2 = min(h, int((y_c + bh/2) * h))
                    if x2 > x1 and y2 > y1:
                        crop_img = img[y1:y2, x1:x2]
                        crop_path = os.path.join(corps, actual_name)
                        cv2.imwrite(crop_path, crop_img)
                        processed_data.append((crop_path, text_label))
                    break
            except ValueError:
                continue
print(f"\n Successfully extracted {len(processed_data)} courtesy amount patches.")
if missing_files>0:
    print(f"{missing_files} files missing because the image or box was not found.")

100%|██████████| 1799/1799 [00:08<00:00, 212.38it/s]


 Successfully extracted 1785 courtesy amount patches.
14 files missing because the image or box was not found.


In [31]:
allN= "0123456789"
char2idx= {char: idx + 1 for idx, char in enumerate(allN)}
idx2char= {idx + 1: char for idx, char in enumerate(allN)}
BLANK_IDX= 0
NUM_CLASSES= len(allN) + 1

class CourtesyDataset(Dataset):
    def __init__(self, data_list, transform=None):
        self.data_list = data_list
        self.transform = transform

    def __len__(self):
        return len(self.data_list)

    def __getitem__(self, idx):
        img_path, text = self.data_list[idx]
        img = Image.open(img_path).convert('L')
        if self.transform:
            img = self.transform(img)

        target = [char2idx[c] for c in text if c in char2idx]
        return img, torch.tensor(target, dtype=torch.long), text, img_path
transform = T.Compose([
    T.Resize((32, 128)),
    T.RandomRotation(degrees=5, fill=255),
    T.ColorJitter(brightness=0.2, contrast=0.2),
    T.ToTensor(),
    T.Normalize(0.5, 0.5)
])
processed_data.sort(key=lambda x: x[0])
random.seed(42)
random.shuffle(processed_data)
split_idx = int(len(processed_data) * 0.8)
train_dataset = CourtesyDataset(processed_data[:split_idx], transform=transform)
val_dataset = CourtesyDataset(processed_data[split_idx:], transform=transform)
def collate_fn(batch):
    images, targets, texts, paths = zip(*batch)
    images = torch.stack(images)
    target_lengths = torch.tensor([len(t) for t in targets], dtype=torch.long)
    targets = torch.cat(targets)
    return images, targets, target_lengths, texts, paths
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)
class CRNN(nn.Module):
    def __init__(self, num_classes):
        super(CRNN, self).__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 64, 3, 1, 1), nn.ReLU(), nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, 3, 1, 1), nn.ReLU(), nn.MaxPool2d(2, 2),
            nn.Conv2d(128, 256, 3, 1, 1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.MaxPool2d((2, 2), (2, 1), (0, 1)),
            nn.Conv2d(256, 512, 3, 1, 1), nn.BatchNorm2d(512), nn.ReLU()
        )
        self.rnn = nn.LSTM(512 * 4, 256, bidirectional=True, num_layers=2, batch_first=True)
        self.fc = nn.Linear(256 * 2, num_classes)

    def forward(self, x):
        conv = self.cnn(x)
        b, c, h, w = conv.size()
        conv = conv.view(b, c * h, w)
        conv = conv.permute(0, 2, 1)
        rnn_out, _ = self.rnn(conv)
        output = self.fc(rnn_out)
        return torch.nn.functional.log_softmax(output, dim=2)

In [32]:
device= torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)
    torch.backends.cudnn.deterministic = True
model= CRNN(NUM_CLASSES).to(device)
criterion= nn.CTCLoss(blank=BLANK_IDX, zero_infinity=True)
optimizer= torch.optim.Adam(model.parameters(), lr=0.001)
scheduler= torch.optim.lr_scheduler.StepLR(optimizer, step_size=15, gamma=0.5)
def decode_predictions(predictions):
    pred_strings= []
    for i in range(predictions.size(0)):
        pred, prev_char= "", BLANK_IDX
        for t in range(predictions.size(1)):
            char_idx= predictions[i, t].item()
            if char_idx!=BLANK_IDX and char_idx != prev_char:
                pred+= idx2char[char_idx]
            prev_char= char_idx
        pred_strings.append(pred)
    return pred_strings
epochs= 40
print(f"Starting Training on {device}...")

for epoch in range(epochs):
    model.train()
    total_loss= 0
    for images, targets, target_lengths, _, _ in train_loader:
        images, targets = images.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs= model(images)
        outputs= outputs.permute(1, 0, 2)
        input_lengths= torch.full(size=(outputs.size(1),), fill_value=outputs.size(0), dtype=torch.long)
        loss= criterion(outputs, targets, input_lengths, target_lengths)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    scheduler.step()
    current_lr= scheduler.get_last_lr()[0]
    print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss/len(train_loader):.4f}")


model.eval()
total_N, total_errors= 0, 0
err_counts = {0: 0, 1: 0, "2+": 0}
output_results= []

with torch.no_grad():
    for images, _, _, texts, paths in val_loader:
        images = images.to(device)
        outputs = model(images)
        _, preds = outputs.max(2)
        decoded_preds = decode_predictions(preds)
        for pred, gt, path in zip(decoded_preds, texts, paths):
            img_name = os.path.basename(path)
            output_results.append(f"{img_name} {pred}")
            N= len(gt)
            if N==0: continue
            err= editdistance.eval(pred, gt)
            total_N+= N
            total_errors+= err
            if err==0: err_counts[0]+= 1
            elif err==1: err_counts[1]+= 1
            else: err_counts["2+"]+= 1
total_samples = sum(err_counts.values())
digit_accuracy = (1 - (total_errors / total_N)) * 100 if total_N > 0 else 0 #(1-(I+D+S)/N)*100

print("Courtesy Amount Recognition Mertics:")
print(f"1. Accuracy at the digit level: {max(0, digit_accuracy):.2f}%")
print(f"2. Percentage of amounts with no errors: {(err_counts[0]/total_samples)*100:.2f}%")
print(f"3. Percentage of amounts with one error: {(err_counts[1]/total_samples)*100:.2f}%")
print(f"4. Percentage of amounts with two or more errors: {(err_counts['2+']/total_samples)*100:.2f}%")

out_path= "PartB_Results.txt"
with open(out_path, 'w') as f:
    f.write("\n".join(output_results))

Starting Training on cuda...
Epoch 1/40 | Loss: 2.8676
Epoch 2/40 | Loss: 2.3028
Epoch 3/40 | Loss: 2.1808
Epoch 4/40 | Loss: 2.1126
Epoch 5/40 | Loss: 2.0059
Epoch 6/40 | Loss: 1.9160
Epoch 7/40 | Loss: 1.7461
Epoch 8/40 | Loss: 1.5017
Epoch 9/40 | Loss: 1.0689
Epoch 10/40 | Loss: 0.6831
Epoch 11/40 | Loss: 0.4559
Epoch 12/40 | Loss: 0.3601
Epoch 13/40 | Loss: 0.3066
Epoch 14/40 | Loss: 0.2341
Epoch 15/40 | Loss: 0.2398
Epoch 16/40 | Loss: 0.1558
Epoch 17/40 | Loss: 0.1197
Epoch 18/40 | Loss: 0.1088
Epoch 19/40 | Loss: 0.0968
Epoch 20/40 | Loss: 0.0941
Epoch 21/40 | Loss: 0.0819
Epoch 22/40 | Loss: 0.0777
Epoch 23/40 | Loss: 0.0772
Epoch 24/40 | Loss: 0.0653
Epoch 25/40 | Loss: 0.0641
Epoch 26/40 | Loss: 0.0587
Epoch 27/40 | Loss: 0.0640
Epoch 28/40 | Loss: 0.0512
Epoch 29/40 | Loss: 0.0534
Epoch 30/40 | Loss: 0.0400
Epoch 31/40 | Loss: 0.0306
Epoch 32/40 | Loss: 0.0254
Epoch 33/40 | Loss: 0.0209
Epoch 34/40 | Loss: 0.0177
Epoch 35/40 | Loss: 0.0185
Epoch 36/40 | Loss: 0.0141
Epoch 37

**PART C**

In [33]:
# =========================
# PATHS
# =========================

extract_path = '/content/ExtractedData'

images_dir = os.path.join(extract_path, 'CheckImages')
labels_dir = os.path.join(extract_path, 'BoundingBoxes')

legal_txt_dir = os.path.join(extract_path, 'LegalAmounts')

legal_crops_dir = "/content/Legal_Crops"
os.makedirs(legal_crops_dir, exist_ok=True)

# =========================
# READ LEGAL AMOUNT LABELS
# =========================

amount_dict = {}

txt_files = glob.glob(os.path.join(legal_txt_dir, '*.txt'))

for txt_file in txt_files:

    with open(txt_file, 'r', encoding='utf-8') as f:

        for line in f:

            if not line.strip():
                continue

            parts = line.strip().split(maxsplit=1)

            if len(parts) < 2:
                continue

            filename = parts[0].strip()
            token_str = parts[1].strip()

            tokens = token_str.strip('[]')

            tokens = tokens.replace("'", "")

            tokens = [t.strip() for t in tokens.split(',')]

            clean_text = " ".join(tokens)
            clean_text = re.sub(r'[\[\]#/⁄]', '', clean_text)

            clean_text = clean_text.replace('\u202b', '')
            clean_text = clean_text.replace('\u202c', '')

            clean_text = re.sub(r'\s+', ' ', clean_text).strip()

            amount_dict[filename] = clean_text[::-1]

print("Total legal labels:", len(amount_dict))

# =========================
# EXTRACT LEGAL REGION CROPS
# =========================

processed_data = []
missing_files = 0

for raw_name, text_label in tqdm(amount_dict.items()):

    actual_name = raw_name
    if actual_name.startswith('Lac'):
        actual_name = 'ac' + actual_name[3:]

    img_path = os.path.join(images_dir, actual_name)

    bbox_name = os.path.splitext(actual_name)[0] + '.txt'
    bbox_path = os.path.join(labels_dir, bbox_name)

    if not os.path.exists(img_path):

        base = os.path.splitext(actual_name)[0]

        for ext in ['.png', '.jpg', '.jpeg']:

            alt = os.path.join(images_dir, base + ext)

            if os.path.exists(alt):

                img_path = alt
                actual_name = base + ext

                break

    if not os.path.exists(img_path) or not os.path.exists(bbox_path):

        missing_files += 1
        continue

    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

    if img is None:
        continue

    h, w = img.shape

    with open(bbox_path, 'r') as f:

        for line in f:

            parts = line.strip().split()

            if not parts:
                continue

            try:

                class_id = int(float(parts[0]))

                # LEGAL AMOUNT = CLASS 0
                if class_id == 0:

                    x_c, y_c, bw, bh = [float(x) for x in parts[1:]]

                    x1 = max(0, int((x_c - bw/2) * w))
                    y1 = max(0, int((y_c - bh/2) * h))
                    x2 = min(w, int((x_c + bw/2) * w))
                    y2 = min(h, int((y_c + bh/2) * h))

                    if x2 > x1 and y2 > y1:

                        crop_img = img[y1:y2, x1:x2]

                        crop_path = os.path.join(legal_crops_dir, actual_name)

                        cv2.imwrite(crop_path, crop_img)

                        processed_data.append((crop_path, text_label))

                    break

            except:
                continue

print(f"\nSuccessfully extracted {len(processed_data)} legal amount patches.")

if missing_files > 0:
    print(f"{missing_files} files were missing.")

# =========================
# BUILD CHARACTER VOCAB
# =========================

all_text = ""

for _, txt in processed_data:
    all_text += txt

vocab = sorted(list(set(all_text)))

print("Vocabulary:")
print(vocab)

char2idx = {char: idx + 1 for idx, char in enumerate(vocab)}
idx2char = {idx + 1: char for idx, char in enumerate(vocab)}

BLANK_IDX = 0
NUM_CLASSES = len(vocab) + 1

print("Number of classes:", NUM_CLASSES)

# =========================
# DATASET
# =========================

class LegalDataset(Dataset):

    def __init__(self, data_list, transform=None):

        self.data_list = data_list
        self.transform = transform

    def __len__(self):
        return len(self.data_list)

    def __getitem__(self, idx):

        img_path, text = self.data_list[idx]

        img = Image.open(img_path).convert('L')

        if self.transform:
            img = self.transform(img)

        target = [char2idx[c] for c in text if c in char2idx]

        return img, torch.tensor(target, dtype=torch.long), text, img_path

# =========================
# TRANSFORMS
# =========================

transform = T.Compose([

    T.Resize((64, 512)),

    T.RandomRotation(
        degrees=2,
        fill=255
    ),

    T.RandomAffine(
        degrees=0,
        translate=(0.02, 0.02),
        scale=(0.95, 1.05),
        fill=255
    ),

    T.ToTensor(),

    T.Normalize((0.5,), (0.5,))
])

# =========================
# TRAIN / VAL SPLIT
# =========================

processed_data.sort(key=lambda x: x[0])

random.seed(42)
random.shuffle(processed_data)

split_idx = int(len(processed_data) * 0.8)

train_dataset = LegalDataset(
    processed_data[:split_idx],
    transform=transform
)

val_dataset = LegalDataset(
    processed_data[split_idx:],
    transform=transform
)

Total legal labels: 1799


100%|██████████| 1799/1799 [00:08<00:00, 200.40it/s]



Successfully extracted 1798 legal amount patches.
1 files were missing.
Vocabulary:
[' ', '0', '1', '4', '6', '،', 'ء', 'آ', 'أ', 'إ', 'ئ', 'ا', 'ب', 'ة', 'ت', 'ث', 'ج', 'ح', 'خ', 'د', 'ر', 'س', 'ش', 'ص', 'ض', 'ط', 'ظ', 'ع', 'غ', 'ف', 'ق', 'ك', 'ل', 'م', 'ن', 'ه', 'و', 'ى', 'ي', 'ً', '٠', '١', '٢', '٣', '٤', '٥', '٦', '٧', '٨', '٩']
Number of classes: 51


In [34]:
# =========================
# COLLATE FUNCTION
# =========================

def collate_fn(batch):

    images, targets, texts, paths = zip(*batch)

    images = torch.stack(images)

    target_lengths = torch.tensor(
        [len(t) for t in targets],
        dtype=torch.long
    )

    targets = torch.cat(targets)

    return images, targets, target_lengths, texts, paths

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    collate_fn=collate_fn
)

val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    shuffle=False,
    collate_fn=collate_fn
)

# =========================
# CRNN MODEL
# =========================

class CRNN(nn.Module):

    def __init__(self, num_classes):

        super(CRNN, self).__init__()

        self.cnn = nn.Sequential(

            nn.Conv2d(1, 64, 3, 1, 1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(64, 128, 3, 1, 1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(128, 256, 3, 1, 1),
            nn.BatchNorm2d(256),
            nn.ReLU(),

            nn.MaxPool2d((2,2), (2,1), (0,1)),

            nn.Conv2d(256, 512, 3, 1, 1),
            nn.BatchNorm2d(512),
            nn.ReLU()

        )

        self.rnn = nn.LSTM(
            512 * 8,
            512,
            bidirectional=True,
            num_layers=2,
            batch_first=True
        )

        self.fc = nn.Linear(512 * 2, num_classes)

    def forward(self, x):

        conv = self.cnn(x)

        b, c, h, w = conv.size()

        conv = conv.view(b, c * h, w)

        conv = conv.permute(0, 2, 1)

        rnn_out, _ = self.rnn(conv)

        output = self.fc(rnn_out)

        return torch.nn.functional.log_softmax(output, dim=2)

# =========================
# DEVICE
# =========================

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)
    torch.backends.cudnn.deterministic = True

model = CRNN(NUM_CLASSES).to(device)

criterion = nn.CTCLoss(
    blank=BLANK_IDX,
    zero_infinity=True
)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=5
)
# =========================
# DECODE FUNCTION
# =========================

def decode_predictions(predictions):

    pred_strings = []

    for i in range(predictions.size(0)):

        pred = ""

        prev_char = BLANK_IDX

        for t in range(predictions.size(1)):

            char_idx = predictions[i, t].item()

            if char_idx != BLANK_IDX and char_idx != prev_char:

                pred += idx2char[char_idx]

            prev_char = char_idx

        pred_strings.append(pred)

    return pred_strings

# =========================
# TRAINING
# =========================

epochs = 80

print(f"Starting training on {device}...")

for epoch in range(epochs):

    model.train()

    total_loss = 0

    for images, targets, target_lengths, _, _ in train_loader:

        images = images.to(device)
        targets = targets.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        outputs = outputs.permute(1, 0, 2)

        input_lengths = torch.full(
            size=(outputs.size(1),),
            fill_value=outputs.size(0),
            dtype=torch.long
        )

        loss = criterion(
            outputs,
            targets,
            input_lengths,
            target_lengths
        )

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    scheduler.step(total_loss)

    print(
        f"Epoch {epoch+1}/{epochs} | "
        f"Loss: {total_loss/len(train_loader):.4f}"
    )

# =========================
# EVALUATION
# =========================

model.eval()

total_char_errors = 0
total_chars = 0

total_word_errors = 0
total_words = 0

output_results = []

with torch.no_grad():

    for images, _, _, texts, paths in val_loader:

        images = images.to(device)

        outputs = model(images)

        _, preds = outputs.max(2)

        decoded_preds = decode_predictions(preds)

        for pred, gt, path in zip(decoded_preds, texts, paths):

            img_name = os.path.basename(path)

            output_results.append(f"{img_name} {pred[::-1]}")

            # =====================
            # CER
            # =====================

            char_errors = editdistance.eval(pred, gt)

            total_char_errors += char_errors
            total_chars += len(gt)

            # =====================
            # WER
            # =====================

            gt_words = gt.split()
            pred_words = pred.split()

            word_errors = editdistance.eval(pred_words, gt_words)

            total_word_errors += word_errors
            total_words += len(gt_words)

# =========================
# FINAL METRICS
# =========================

CER = (total_char_errors / total_chars) * 100 if total_chars > 0 else 0

WER = (total_word_errors / total_words) * 100 if total_words > 0 else 0

print("\n=========================")
print("LEGAL AMOUNT RECOGNITION")
print("=========================")

print(f"CER (Character Error Rate): {CER:.2f}%")

print(f"WER (Word Error Rate): {WER:.2f}%")

# =========================
# SAVE RESULTS
# =========================

out_path = "PartC_Results.txt"

with open(out_path, 'w', encoding='utf-8') as f:

    f.write("\n".join(output_results))

print(f"\nResults saved to: {out_path}")

Starting training on cuda...
Epoch 1/80 | Loss: 2.8672
Epoch 2/80 | Loss: 2.5346
Epoch 3/80 | Loss: 2.3715
Epoch 4/80 | Loss: 2.2845
Epoch 5/80 | Loss: 2.1901
Epoch 6/80 | Loss: 2.0897
Epoch 7/80 | Loss: 1.9881
Epoch 8/80 | Loss: 1.9283
Epoch 9/80 | Loss: 1.9091
Epoch 10/80 | Loss: 1.8534
Epoch 11/80 | Loss: 1.8697
Epoch 12/80 | Loss: 1.8458
Epoch 13/80 | Loss: 1.8405
Epoch 14/80 | Loss: 1.7696
Epoch 15/80 | Loss: 1.7531
Epoch 16/80 | Loss: 1.7157
Epoch 17/80 | Loss: 1.6412
Epoch 18/80 | Loss: 1.6019
Epoch 19/80 | Loss: 1.5617
Epoch 20/80 | Loss: 1.5459
Epoch 21/80 | Loss: 1.5281
Epoch 22/80 | Loss: 1.4944
Epoch 23/80 | Loss: 1.5672
Epoch 24/80 | Loss: 1.5116
Epoch 25/80 | Loss: 1.4784
Epoch 26/80 | Loss: 1.4564
Epoch 27/80 | Loss: 1.4092
Epoch 28/80 | Loss: 1.4079
Epoch 29/80 | Loss: 1.3926
Epoch 30/80 | Loss: 1.3540
Epoch 31/80 | Loss: 1.3157
Epoch 32/80 | Loss: 1.3069
Epoch 33/80 | Loss: 1.2821
Epoch 34/80 | Loss: 1.2448
Epoch 35/80 | Loss: 1.2358
Epoch 36/80 | Loss: 1.2411
Epoch 37

**PART D**

In [35]:
courtesy_results = {}
with open("PartB_Results.txt", "r", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split(maxsplit=1)
        if len(parts) == 2:
            courtesy_results[parts[0]] = parts[1]

legal_results = {}
with open("PartC_Results.txt", "r", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split(maxsplit=1)
        if len(parts) == 2:
            legal_results[parts[0]] = parts[1]

def int_to_arabic_words(num_str):
    try:
        n = int(num_str)
    except ValueError:
        return []

    if n == 0:
        return ["صفر", "ريال"]

    units = ["", "واحد", "اثنان", "ثلاثة", "أربعة", "خمسة", "ستة", "سبعة", "ثمانية", "تسعة"]
    teens = ["عشرة", "أحد عشر", "اثنا عشر", "ثلاثة عشر", "أربعة عشر", "خمسة عشر", "ستة عشر", "سبعة عشر", "ثمانية عشر", "تسعة عشر"]
    tens = ["", "عشرة", "عشرون", "ثلاثون", "أربعون", "خمسون", "ستون", "سبعون", "ثمانون", "تسعون"]
    hundreds = ["", "مائة", "مائتان", "ثلاثمائة", "أربعمائة", "خمسمائة", "ستمائة", "سبعمائة", "ثمانمائة", "تسعمائة"]

    def get_parts(val):
        res = []
        h = val // 100
        rem = val % 100
        if h > 0: res.append(hundreds[h])
        if rem > 0:
            if h > 0: res.append("و")
            if rem < 10: res.append(units[rem])
            elif 10 <= rem < 20: res.extend(teens[rem - 10].split())
            else:
                t = rem // 10
                u = rem % 10
                if u > 0:
                    res.append(units[u])
                    res.append("و")
                res.append(tens[t])
        return res

    words = []
    millions = n // 1000000
    n = n % 1000000
    thousands = n // 1000
    rem = n % 1000

    if millions > 0:
        if millions == 1: words.append("مليون")
        elif millions == 2: words.append("مليونان")
        elif 3 <= millions <= 10:
            words.extend(get_parts(millions))
            words.append("ملايين")
        else:
            words.extend(get_parts(millions))
            words.append("مليون")

    if thousands > 0:
        if words: words.append("و")
        if thousands == 1: words.append("ألف")
        elif thousands == 2: words.append("ألفان")
        elif 3 <= thousands <= 10:
            words.extend(get_parts(thousands))
            words.append("آلاف")
        else:
            words.extend(get_parts(thousands))
            words.append("ألف")

    if rem > 0:
        if words: words.append("و")
        words.extend(get_parts(rem))

    words.append("ريال")
    return [w for w in words if w]

def normalize_arabic(text):
    text = re.sub(r'[أإآ]', 'ا', text)
    text = re.sub(r'ة', 'ه', text)
    text = re.sub(r'ى', 'ي', text)
    return text

def tokenize_and_normalize_ocr(text):
    text = re.sub(r'(فقط|لاغير)', '', text)
    text = re.sub(r'[^\u0600-\u06FF\s]', ' ', text)
    return [normalize_arabic(t) for t in text.split()]

verification_results = []
verified_count = 0
failed_count = 0
overlapping_files= list(set(courtesy_results.keys()).intersection(set(legal_results.keys())))

print(f"Evaluating {len(overlapping_files)} overlapping validation files...\n")

for filename in overlapping_files:
    courtesy_num = courtesy_results[filename]
    legal_text = legal_results[filename]

    expected_tokens = [normalize_arabic(t) for t in int_to_arabic_words(courtesy_num)]
    predicted_tokens = tokenize_and_normalize_ocr(legal_text)

    expected_str = "".join(expected_tokens)
    predicted_str = "".join(predicted_tokens)
    noise_to_remove = ["لاغير", "لغير", "فقط", "فير", "افقر", "الاغير", "وقد"]
    for noise in noise_to_remove:
        predicted_str = predicted_str.replace(noise, "")
        expected_str = expected_str.replace(noise, "")

    error = editdistance.eval(expected_str, predicted_str)
    max_len = max(len(expected_str), len(predicted_str), 1)
    similarity = 1 - (error / max_len)
    verified = similarity >= 0.50

    if verified:
        status = "VERIFIED"
        verified_count += 1
    else:
        status = "FAILED"
        failed_count += 1

    verification_results.append(
        f"{filename} | C_Num: {courtesy_num} | Sim: {similarity:.2f} | {status} \n"
        f"   Expected: {expected_str}\n"
        f"   Predicted: {predicted_str}\n"
    )
total = verified_count + failed_count
accuracy = (verified_count / total * 100) if total > 0 else 0

print("=========================")
print("FINAL VERIFICATION METRICS")
print("=========================")
print(f"Verified Checks: {verified_count}")
print(f"Failed Checks: {failed_count}")
print(f"Verification Accuracy: {accuracy:.2f}%\n")

out_path = "PartD_Verification.txt"
with open(out_path, "w", encoding="utf-8") as f:
    f.write("\n".join(verification_results))

Evaluating 289 overlapping validation files...

FINAL VERIFICATION METRICS
Verified Checks: 227
Failed Checks: 62
Verification Accuracy: 78.55%

